# 03A — Target-design experiments

This notebook runs the evidence needed **before** finalizing the household-day target. It evaluates (A) target-day coverage through empirical masking of complete days, (B) separate target-day and history-day coverage rules, and (C) rolling household-relative P95 thresholds over 30, 45, and 60 calendar days.

No model is trained, no model score is used to select a target, and no final household-day or model-ready dataset is written. The candidate configuration remains explicitly **unfrozen** for researcher review.

**Empirical masking:** actual valid-slot omissions are reconstructed for 92–95/96 days from the preprocessed interval CSV. Those observed missing-slot patterns are sampled and applied to every available 96/96 reference day. The full and masked peaks are compared against the *same* past-only rolling P95 threshold. Five seeded replications quantify pattern-sampling variability.

**Rolling windows:** historical peaks use only dates strictly before the target date. A history day contributes only when it meets the tested history-day interval threshold; a target day must independently meet its tested target-day threshold. This keeps target-day and history-day completeness separate.

Inputs are the validated preprocessing CSV, its household-day coverage audit, and existing interval-EDA CSV audits. All tabular outputs are CSV; no Parquet input/output is used.

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import json
import math
import hashlib
import csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.5f}")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent
if not (PROJECT_ROOT / "data").exists():
    raise RuntimeError(f"Could not locate project root from {CURRENT_DIR}")

DATA_DIR = PROJECT_ROOT / "data"
INTERVAL_PATH = DATA_DIR / "processed" / "01_preprocessed_lp_interval_data.csv"
PREPROCESSING_MANIFEST_PATH = DATA_DIR / "processed" / "01_preprocessing_manifest.json"
COVERAGE_PATH = DATA_DIR / "interim" / "01_household_day_coverage_audit.csv"
COMPLETE_DAY_EDA_PATH = PROJECT_ROOT / "reports" / "tables" / "02_interval_eda" / "02_05_complete_day_shape_metrics.csv"
MISSING_SLOT_EDA_PATH = PROJECT_ROOT / "reports" / "tables" / "02_interval_eda" / "02_05_household_day_missing_slot_patterns.csv"
EDA_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "02_interval_eda" / "02_eda_manifest.json"

TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "03a_target_design_experiments"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "03a_target_design_experiments"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MASKING_RESULTS_PATH = TABLE_DIR / "03a_completeness_masking_results.csv"
TARGET_RESULTS_PATH = TABLE_DIR / "03a_target_design_results.csv"
CONFIG_RESULTS_PATH = TABLE_DIR / "03a_selected_target_config.json"
VALIDATION_PATH = TABLE_DIR / "03a_validation_checks.csv"
FIGURE_PATH = FIGURE_DIR / "03a_target_design_summary.png"
TARGET_EVIDENCE_FIGURE_PATH = FIGURE_DIR / "03a_target_day_evidence.png"
HISTORY_POLICY_FIGURE_PATH = FIGURE_DIR / "03a_history_policy_heatmaps.png"
HISTORY_PARETO_FIGURE_PATH = FIGURE_DIR / "03a_history_policy_pareto.png"
HISTORY_SUPPORT_FIGURE_PATH = FIGURE_DIR / "03a_historical_support_histogram.png"
HISTORY_TRAJECTORY_FIGURE_PATH = FIGURE_DIR / "03a_rolling_p95_household_trajectories.png"
HISTORY_POLICY_RESULTS_PATH = TABLE_DIR / "03a_history_quality_support_experiment.csv"
HISTORY_SUPPORT_DISTRIBUTION_PATH = TABLE_DIR / "03a_historical_support_distribution.csv"

EXPECTED_INTERVALS = 96
TARGET_COVERAGE_CANDIDATES = [96, 95, 94, 93, 92]
HISTORY_COVERAGE_CANDIDATES = [96, 95, 94, 93, 92]
HISTORICAL_WINDOWS_DAYS = [30, 45, 60]
MINIMUM_HISTORY_FRACTIONS = [0.70, 0.75, 0.80]
HISTORY_FRACTION_FOR_CEIL = True
PRIMARY_QUANTILE = 0.95
QUANTILE_METHOD = "linear"
MIN_COMPLETE_HISTORY_DAYS_FOR_MASK_LABEL = 45
MASK_REPLICATES = 5
RANDOM_SEED = 42
CSV_CHUNK_SIZE = 250_000
MASK_BATCH_SIZE = 25_000

required_inputs = [
    INTERVAL_PATH, PREPROCESSING_MANIFEST_PATH, COVERAGE_PATH,
    COMPLETE_DAY_EDA_PATH, MISSING_SLOT_EDA_PATH, EDA_MANIFEST_PATH,
]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Required validated inputs are missing: {missing_inputs}")

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_csv_atomic(frame, path):
    temporary_path = Path(path).with_suffix(".tmp.csv")
    frame.to_csv(temporary_path, index=False, encoding="utf-8", na_rep="", float_format="%.12g")
    temporary_path.replace(path)

def write_json_atomic(payload, path):
    temporary_path = Path(path).with_suffix(Path(path).suffix + ".tmp")
    temporary_path.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
    temporary_path.replace(path)

def file_state(path):
    stat = Path(path).stat()
    return {"path": str(Path(path).resolve()), "size_bytes": int(stat.st_size), "modified_ns": int(stat.st_mtime_ns)}

preprocessing_manifest = read_json(PREPROCESSING_MANIFEST_PATH)
eda_manifest = read_json(EDA_MANIFEST_PATH)
if not all(preprocessing_manifest.get("validation", {}).values()):
    raise RuntimeError("The preprocessing manifest does not report all checks as passed.")
if not all(eda_manifest.get("validation", {}).values()):
    raise RuntimeError("The interval EDA manifest does not report all checks as passed.")
if preprocessing_manifest.get("policy", {}).get("timezone") != "Asia/Colombo":
    raise RuntimeError("The current preprocessing manifest does not confirm Asia/Colombo local dates.")

INTERVAL_INPUT_STATE_BEFORE = file_state(INTERVAL_PATH)
source_signature = preprocessing_manifest["primary_signature"]
if INTERVAL_INPUT_STATE_BEFORE["size_bytes"] != int(source_signature["size_bytes"]):
    raise RuntimeError("Preprocessed interval CSV size differs from its validated manifest.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Preprocessed interval input: {INTERVAL_INPUT_STATE_BEFORE['size_bytes']:,} bytes")
print(f"Upstream row count: {int(source_signature['rows']):,}")
print(f"Outputs: {TABLE_DIR}")

## Load and validate the upstream CSV evidence

The complete-day EDA supplies the trusted 96/96 reference peaks. The preprocessing coverage audit defines each household-day's valid-slot count. The full interval CSV is then streamed once to independently rebuild daily valid peaks, collect the 96 slot values for complete days, and recover exact missing-slot patterns for 92–95/96 days.

In [ ]:
coverage_columns = [
    "household_id", "measurement_date", "valid_interval_count", "coverage_ratio",
    "is_complete_96_day",
]
coverage = pd.read_csv(
    COVERAGE_PATH,
    usecols=coverage_columns,
    dtype={
        "household_id": "string",
        "valid_interval_count": "int16",
        "coverage_ratio": "float32",
        "is_complete_96_day": "boolean",
    },
    true_values=["True"], false_values=["False"],
    parse_dates=["measurement_date"],
    keep_default_na=False, na_values=[""], encoding="utf-8", low_memory=False,
)
coverage["measurement_date"] = pd.to_datetime(coverage["measurement_date"], errors="raise").dt.normalize()
coverage["household_id"] = coverage["household_id"].astype("string")
coverage = coverage.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True)
if coverage.duplicated(["household_id", "measurement_date"]).any():
    raise RuntimeError("Coverage audit has duplicate household-date keys.")
if not coverage["valid_interval_count"].between(0, EXPECTED_INTERVALS).all():
    raise RuntimeError("Coverage audit contains interval counts outside 0..96.")
if not coverage["is_complete_96_day"].eq(coverage["valid_interval_count"].eq(EXPECTED_INTERVALS)).all():
    raise RuntimeError("The complete-day flag disagrees with the interval count.")

shape = pd.read_csv(
    COMPLETE_DAY_EDA_PATH,
    usecols=["household_id", "measurement_date", "valid_interval_count", "daily_peak_kw"],
    dtype={"household_id": "string", "valid_interval_count": "int16", "daily_peak_kw": "float64"},
    parse_dates=["measurement_date"],
    keep_default_na=False, na_values=[""], encoding="utf-8", low_memory=False,
)
shape["measurement_date"] = pd.to_datetime(shape["measurement_date"], errors="raise").dt.normalize()
shape["household_id"] = shape["household_id"].astype("string")
shape = shape.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True)
if shape.duplicated(["household_id", "measurement_date"]).any():
    raise RuntimeError("Complete-day EDA table has duplicate household-date keys.")
if not shape["valid_interval_count"].eq(EXPECTED_INTERVALS).all():
    raise RuntimeError("Complete-day EDA table contains a non-96/96 row.")
if shape["daily_peak_kw"].isna().any() or not np.isfinite(shape["daily_peak_kw"]).all():
    raise RuntimeError("Complete-day EDA peaks contain missing or non-finite values.")

coverage_complete_count = int(coverage["is_complete_96_day"].sum())
if len(shape) != coverage_complete_count:
    raise RuntimeError(
        f"Complete-day EDA rows ({len(shape):,}) != coverage complete rows ({coverage_complete_count:,})."
    )
print(f"Observed household-days in coverage audit: {len(coverage):,}")
print(f"Complete 96/96 reference days: {len(shape):,}")
print(f"Households: {coverage['household_id'].nunique():,}")
display(coverage["valid_interval_count"].value_counts().sort_index().rename_axis("valid_intervals").reset_index(name="household_days"))

In [ ]:
# Reconstruct interval-level peaks and exact missing-slot patterns in one CSV pass.
# This does not modify the source interval file or write a derived interval dataset.
households = pd.Index(coverage["household_id"].drop_duplicates().astype(str))
household_code = {household_id: code for code, household_id in enumerate(households)}
coverage_house_codes = coverage["household_id"].astype(str).map(household_code).to_numpy(dtype=np.int32)
coverage_day_codes = coverage["measurement_date"].values.astype("datetime64[D]").astype(np.int32)
coverage_keys = coverage_house_codes.astype(np.int64) * 100_000 + coverage_day_codes.astype(np.int64)
coverage_key_index = pd.Index(coverage_keys, name="household_day_key")
if not coverage_key_index.is_unique:
    raise RuntimeError("Internal household-day key encoding is not unique.")

shape_house_codes = shape["household_id"].astype(str).map(household_code).to_numpy(dtype=np.int32)
shape_day_codes = shape["measurement_date"].values.astype("datetime64[D]").astype(np.int32)
shape_coverage_rows = coverage_key_index.get_indexer(
    shape_house_codes.astype(np.int64) * 100_000 + shape_day_codes.astype(np.int64)
)
if (shape_coverage_rows < 0).any():
    raise RuntimeError("Some EDA complete-day keys are absent from the coverage audit.")
if not coverage.loc[shape_coverage_rows, "valid_interval_count"].eq(EXPECTED_INTERVALS).all():
    raise RuntimeError("A complete-day EDA row maps to non-complete coverage.")

complete_position_by_coverage = np.full(len(coverage), -1, dtype=np.int32)
complete_position_by_coverage[shape_coverage_rows] = np.arange(len(shape), dtype=np.int32)
candidate_pattern_day = coverage["valid_interval_count"].between(92, 95).to_numpy()
valid_count_expected = coverage["valid_interval_count"].to_numpy(dtype=np.uint16)

observed_valid_count = np.zeros(len(coverage), dtype=np.uint16)
observed_peak_kw = np.full(len(coverage), -np.inf, dtype=np.float64)
complete_slot_values = np.full((len(shape), EXPECTED_INTERVALS), np.nan, dtype=np.float64)
candidate_valid_slots = defaultdict(set)

interval_columns = [
    "household_id", "measurement_date", "time_slot_index",
    "avg_import_kw", "is_valid_for_daily_aggregation",
]
interval_dtypes = {
    "household_id": "string",
    "measurement_date": "string",
    "time_slot_index": "int16",
    "avg_import_kw": "float64",
    "is_valid_for_daily_aggregation": "boolean",
}
rows_scanned = 0
valid_rows_scanned = 0
households_seen = set()

reader = pd.read_csv(
    INTERVAL_PATH,
    usecols=interval_columns,
    dtype=interval_dtypes,
    true_values=["True"], false_values=["False"],
    chunksize=CSV_CHUNK_SIZE,
    keep_default_na=False, na_values=[""], encoding="utf-8",
    low_memory=False,
)
for frame in reader:
    rows_scanned += len(frame)
    date_values = pd.to_datetime(frame["measurement_date"], errors="raise").values.astype("datetime64[D]").astype(np.int32)
    household_values = frame["household_id"].astype(str)
    house_codes = household_values.map(household_code)
    if house_codes.isna().any():
        unknown = household_values[house_codes.isna()].drop_duplicates().head(10).tolist()
        raise RuntimeError(f"Interval data contains household IDs absent from coverage audit: {unknown}")
    row_keys = house_codes.to_numpy(dtype=np.int64) * 100_000 + date_values.astype(np.int64)
    day_rows = coverage_key_index.get_indexer(row_keys)
    if (day_rows < 0).any():
        raise RuntimeError("Interval rows contain household-date keys absent from the coverage audit.")

    valid_mask = frame["is_valid_for_daily_aggregation"].fillna(False).to_numpy(dtype=bool)
    if not valid_mask.any():
        continue
    valid_rows_scanned += int(valid_mask.sum())
    valid_day_rows = day_rows[valid_mask].astype(np.int32)
    valid_slots = frame.loc[valid_mask, "time_slot_index"].to_numpy(dtype=np.int16)
    valid_kw = frame.loc[valid_mask, "avg_import_kw"].to_numpy(dtype=np.float64)
    if np.isnan(valid_kw).any():
        raise RuntimeError("A valid-for-aggregation row has missing avg_import_kw.")
    if ((valid_slots < 0) | (valid_slots >= EXPECTED_INTERVALS)).any():
        raise RuntimeError("A valid interval has a time_slot_index outside 0..95.")

    grouped = pd.DataFrame({"day_row": valid_day_rows, "kw": valid_kw}).groupby(
        "day_row", sort=False, observed=True
    ).agg(valid_count=("kw", "size"), peak_kw=("kw", "max"))
    grouped_rows = grouped.index.to_numpy(dtype=np.int32)
    observed_valid_count[grouped_rows] += grouped["valid_count"].to_numpy(dtype=np.uint16)
    observed_peak_kw[grouped_rows] = np.maximum(
        observed_peak_kw[grouped_rows], grouped["peak_kw"].to_numpy(dtype=np.float64)
    )

    complete_positions = complete_position_by_coverage[valid_day_rows]
    is_complete = complete_positions >= 0
    if is_complete.any():
        complete_slot_values[complete_positions[is_complete], valid_slots[is_complete]] = valid_kw[is_complete]

    is_pattern_candidate = candidate_pattern_day[valid_day_rows]
    if is_pattern_candidate.any():
        candidate_frame = pd.DataFrame({
            "day_row": valid_day_rows[is_pattern_candidate],
            "slot": valid_slots[is_pattern_candidate],
        })
        for day_row, slots in candidate_frame.groupby("day_row", sort=False)["slot"]:
            candidate_valid_slots[int(day_row)].update(int(slot) for slot in slots.tolist())

    # The primary CSV is expected to remain household-sorted, but this audit only
    # needs household-date keys and is robust to chunk boundaries.
    if len(households_seen) < 1:
        households_seen.update(household_values.unique().tolist())

observed_peak_kw[observed_valid_count == 0] = np.nan
if rows_scanned != int(source_signature["rows"]):
    raise RuntimeError(f"Scanned {rows_scanned:,} rows; manifest expects {int(source_signature['rows']):,}.")
expected_valid_rows = int(preprocessing_manifest["counts"]["valid_aggregation_rows"])
if valid_rows_scanned != expected_valid_rows:
    raise RuntimeError(f"Scanned {valid_rows_scanned:,} valid rows; manifest expects {expected_valid_rows:,}.")
if not np.array_equal(observed_valid_count, valid_count_expected):
    mismatches = int(np.count_nonzero(observed_valid_count != valid_count_expected))
    raise RuntimeError(f"Reconstructed valid-slot counts disagree with coverage for {mismatches:,} days.")
if not np.isfinite(complete_slot_values).all():
    raise RuntimeError("At least one 96/96 reference day does not have all 96 slot values.")
streamed_complete_peaks = complete_slot_values.max(axis=1)
if not np.allclose(streamed_complete_peaks, shape["daily_peak_kw"].to_numpy(dtype=np.float64), rtol=0, atol=1e-9):
    raise RuntimeError("Stream-reconstructed complete-day peaks disagree with the existing EDA table.")

pattern_pools = {missing_count: [] for missing_count in range(1, 5)}
pattern_source_day_counts = {}
for day_row in np.flatnonzero(candidate_pattern_day):
    valid_count = int(valid_count_expected[day_row])
    expected_missing = EXPECTED_INTERVALS - valid_count
    valid_slots = candidate_valid_slots.get(int(day_row), set())
    missing_slots = tuple(slot for slot in range(EXPECTED_INTERVALS) if slot not in valid_slots)
    if len(missing_slots) != expected_missing:
        raise RuntimeError(
            f"Missing-slot reconstruction mismatch at coverage row {day_row}: "
            f"expected {expected_missing}, reconstructed {len(missing_slots)}."
        )
    pattern_pools[expected_missing].append(missing_slots)

for missing_count, pool in pattern_pools.items():
    pattern_source_day_counts[missing_count] = len(pool)
    if not pool:
        raise RuntimeError(f"No empirical missing-slot patterns exist for {missing_count} missing intervals.")

print(f"Interval rows scanned: {rows_scanned:,}")
print(f"Valid intervals reconciled to coverage: {valid_rows_scanned:,}")
print(f"Complete-day slot matrix: {complete_slot_values.shape[0]:,} days × 96 slots")
display(pd.DataFrame([
    {"missing_intervals": missing_count, "source_incomplete_days": count}
    for missing_count, count in pattern_source_day_counts.items()
]))

## Build the household calendar and past-only thresholds

The history-day comparison uses the observed daily peak only when that day meets the tested history completeness rule. The target date is excluded from every rolling history calculation. Fully missing calendar days remain in the calendar with zero valid intervals and no peak.

In [ ]:
# Build the continuous household calendar used for valid-time rolling windows.
calendar_parts = []
for household_id, group in coverage.groupby("household_id", sort=False, observed=True):
    dates = pd.date_range(group["measurement_date"].min(), group["measurement_date"].max(), freq="D")
    calendar_parts.append(pd.DataFrame({
        "household_id": household_id,
        "measurement_date": dates,
    }))
calendar = pd.concat(calendar_parts, ignore_index=True)
calendar = calendar.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True)

calendar_house_codes = calendar["household_id"].astype(str).map(household_code).to_numpy(dtype=np.int32)
calendar_day_codes = calendar["measurement_date"].values.astype("datetime64[D]").astype(np.int32)
calendar_keys = calendar_house_codes.astype(np.int64) * 100_000 + calendar_day_codes.astype(np.int64)
calendar_coverage_rows = coverage_key_index.get_indexer(calendar_keys)
calendar["coverage_row"] = calendar_coverage_rows
calendar["valid_interval_count"] = np.zeros(len(calendar), dtype=np.int16)
calendar["daily_peak_kw"] = np.nan
has_observed_day = calendar_coverage_rows >= 0
observed_calendar_rows = np.flatnonzero(has_observed_day)
source_coverage_rows = calendar_coverage_rows[has_observed_day]
calendar.loc[observed_calendar_rows, "valid_interval_count"] = valid_count_expected[source_coverage_rows].astype(np.int16)
calendar.loc[observed_calendar_rows, "daily_peak_kw"] = observed_peak_kw[source_coverage_rows]
calendar["valid_interval_count"] = calendar["valid_interval_count"].astype(np.int16)

# Confirm the continuous calendar adds only unobserved household-days, not duplicates.
if calendar.duplicated(["household_id", "measurement_date"]).any():
    raise RuntimeError("Continuous household calendar contains duplicate keys.")
expected_calendar_rows = int(coverage.groupby("household_id", observed=True)["measurement_date"].agg(lambda dates: (dates.max() - dates.min()).days + 1).sum())
if len(calendar) != expected_calendar_rows:
    raise RuntimeError("Continuous calendar row count disagrees with the independently calculated household date ranges.")

calendar_key_index = pd.Index(calendar_keys, name="household_day_key")
if not calendar_key_index.is_unique:
    raise RuntimeError("Continuous-calendar household-date keys are not unique.")
shape_calendar_rows = calendar_key_index.get_indexer(
    shape_house_codes.astype(np.int64) * 100_000 + shape_day_codes.astype(np.int64)
)
if (shape_calendar_rows < 0).any():
    raise RuntimeError("A complete reference day is absent from the continuous household calendar.")

# Compute past-only P95 thresholds for candidate windows and history-day quality
# rules. The date-offset rolling window is closed on the left: day d is excluded.
thresholds = {}
history_counts = {}
threshold_abs_change = {}
threshold_relative_change = {}
household_calendar_positions = [
    positions for positions in calendar.groupby("household_id", sort=False, observed=True).indices.values()
]
for window_days in HISTORICAL_WINDOWS_DAYS:
    for history_min_intervals in HISTORY_COVERAGE_CANDIDATES:
        threshold_values = np.full(len(calendar), np.nan, dtype=np.float64)
        count_values = np.zeros(len(calendar), dtype=np.uint8)
        for positions in household_calendar_positions:
            group = calendar.iloc[positions]
            history_peak = group["daily_peak_kw"].where(
                group["valid_interval_count"].ge(history_min_intervals)
            )
            rolling = pd.Series(
                history_peak.to_numpy(dtype=np.float64),
                index=pd.DatetimeIndex(group["measurement_date"]),
            ).rolling(f"{window_days}D", closed="left")
            threshold_values[positions] = rolling.quantile(
                PRIMARY_QUANTILE, interpolation=QUANTILE_METHOD
            ).to_numpy(dtype=np.float64)
            count_values[positions] = rolling.count().fillna(0).to_numpy(dtype=np.uint8)
        thresholds[(window_days, history_min_intervals)] = threshold_values
        history_counts[(window_days, history_min_intervals)] = count_values

        abs_change = np.full(len(calendar), np.nan, dtype=np.float64)
        relative_change = np.full(len(calendar), np.nan, dtype=np.float64)
        for positions in household_calendar_positions:
            values = threshold_values[positions]
            previous = np.roll(values, 1)
            previous[0] = np.nan
            abs_change[positions] = np.abs(values - previous)
            relative_change[positions] = np.divide(
                np.abs(values - previous), np.abs(previous),
                out=np.full(len(values), np.nan, dtype=np.float64),
                where=np.isfinite(previous) & (np.abs(previous) > 0),
            )
        threshold_abs_change[(window_days, history_min_intervals)] = abs_change
        threshold_relative_change[(window_days, history_min_intervals)] = relative_change

full_history_p95_60 = thresholds[(60, 96)]
full_history_count_60 = history_counts[(60, 96)]
shape_full_history_p95 = full_history_p95_60[shape_calendar_rows]
shape_full_history_count = full_history_count_60[shape_calendar_rows]
mask_label_eligible = (
    (shape_full_history_count >= MIN_COMPLETE_HISTORY_DAYS_FOR_MASK_LABEL)
    & np.isfinite(shape_full_history_p95)
)
shape_full_peak = complete_slot_values.max(axis=1).astype(np.float64)
shape_full_label = shape_full_peak > shape_full_history_p95
print(f"Continuous household-calendar days: {len(calendar):,}")
print(f"Reference days with a valid past-only P95 label: {int(mask_label_eligible.sum()):,}")
print("Rolling P95 windows completed for 30/45/60 days and 96/95/94/93/92-interval history days.")

## Part A — Empirical masking of complete days

For each 95/94/93/92 threshold, a missing-slot pattern is sampled from an actual incomplete day with exactly that many absent valid intervals. It is applied to each 96/96 reference day. Five deterministic replicates measure the sensitivity to pattern sampling. The baseline is 96/96 with no masking. P95 labels use the same prior-60-calendar-day, complete-history threshold for both full and masked peaks.

In [ ]:
# Part A: mask every full reference day using sampled empirical patterns.
def binary_confusion(reference, candidate):
    reference = np.asarray(reference, dtype=bool)
    candidate = np.asarray(candidate, dtype=bool)
    return {
        "true_positive_labels": int(np.sum(reference & candidate)),
        "true_negative_labels": int(np.sum(~reference & ~candidate)),
        "false_positive_labels": int(np.sum(~reference & candidate)),
        "false_negative_labels": int(np.sum(reference & ~candidate)),
    }

masking_rows = []
rng_master = np.random.default_rng(RANDOM_SEED)
coverage_to_calendar_rows = np.full(len(coverage), -1, dtype=np.int32)
coverage_to_calendar_rows[calendar_coverage_rows[has_observed_day]] = observed_calendar_rows

for target_min_intervals in TARGET_COVERAGE_CANDIDATES:
    missing_count = EXPECTED_INTERVALS - target_min_intervals
    reference_count = len(shape_full_peak)

    potential_rows_recovered = 0
    if target_min_intervals < EXPECTED_INTERVALS:
        incomplete_coverage_rows = np.flatnonzero(
            coverage["valid_interval_count"].ge(target_min_intervals).to_numpy()
            & coverage["valid_interval_count"].lt(EXPECTED_INTERVALS).to_numpy()
        )
        mapped_calendar_rows = coverage_to_calendar_rows[incomplete_coverage_rows]
        potential_rows_recovered = int(np.sum(
            (full_history_count_60[mapped_calendar_rows] >= MIN_COMPLETE_HISTORY_DAYS_FOR_MASK_LABEL)
            & np.isfinite(full_history_p95_60[mapped_calendar_rows])
        ))

    if target_min_intervals == EXPECTED_INTERVALS:
        replicates = [None]
        pattern_array = None
        source_pattern_days = 0
    else:
        source_patterns = np.asarray(pattern_pools[missing_count], dtype=np.int16)
        source_pattern_days = len(source_patterns)
        replicates = range(1, MASK_REPLICATES + 1)

    for replicate in replicates:
        if target_min_intervals == EXPECTED_INTERVALS:
            masked_peaks = shape_full_peak.copy()
        else:
            rng = np.random.default_rng(RANDOM_SEED + target_min_intervals * 100 + int(replicate))
            sampled_pattern_ids = rng.integers(0, len(source_patterns), size=reference_count)
            sampled_patterns = source_patterns[sampled_pattern_ids]
            masked_peaks = np.empty(reference_count, dtype=np.float64)
            for start in range(0, reference_count, MASK_BATCH_SIZE):
                stop = min(start + MASK_BATCH_SIZE, reference_count)
                batch = complete_slot_values[start:stop].copy()
                batch_rows = np.arange(stop - start)
                batch[batch_rows[:, None], sampled_patterns[start:stop]] = -np.inf
                masked_peaks[start:stop] = batch.max(axis=1)

        absolute_error = np.abs(masked_peaks - shape_full_peak)
        relative_error = np.divide(
            absolute_error, np.abs(shape_full_peak),
            out=np.full(reference_count, np.nan, dtype=np.float64),
            where=np.abs(shape_full_peak) > 0,
        )
        masked_labels = masked_peaks > shape_full_history_p95
        confusion = binary_confusion(
            shape_full_label[mask_label_eligible],
            masked_labels[mask_label_eligible],
        )
        confusion["label_evaluable_days"] = int(mask_label_eligible.sum())
        confusion["label_agreement_rate"] = float(np.mean(
            shape_full_label[mask_label_eligible] == masked_labels[mask_label_eligible]
        ))
        confusion["full_positive_labels"] = int(shape_full_label[mask_label_eligible].sum())
        confusion["masked_positive_labels"] = int(masked_labels[mask_label_eligible].sum())

        masking_rows.append({
            "row_type": "replicate",
            "target_min_intervals": target_min_intervals,
            "simulated_missing_intervals": missing_count,
            "replicate": "baseline" if replicate is None else int(replicate),
            "reference_complete_days": reference_count,
            "label_evaluable_days": confusion["label_evaluable_days"],
            "pattern_source_days": source_pattern_days,
            "potential_observed_rows_recovered": potential_rows_recovered,
            "mean_absolute_peak_error_kw": float(np.mean(absolute_error)),
            "median_absolute_peak_error_kw": float(np.median(absolute_error)),
            "p90_absolute_peak_error_kw": float(np.quantile(absolute_error, 0.90)),
            "p95_absolute_peak_error_kw": float(np.quantile(absolute_error, 0.95)),
            "max_absolute_peak_error_kw": float(np.max(absolute_error)),
            "mean_relative_peak_error": float(np.nanmean(relative_error)),
            "median_relative_peak_error": float(np.nanmedian(relative_error)),
            "peak_unchanged_rate": float(np.mean(absolute_error == 0)),
            **confusion,
        })

masking_replicates = pd.DataFrame(masking_rows)
masking_summaries = []
for target_min_intervals, group in masking_replicates.groupby("target_min_intervals", sort=False):
    numeric_metrics = [
        "mean_absolute_peak_error_kw", "median_absolute_peak_error_kw",
        "p90_absolute_peak_error_kw", "p95_absolute_peak_error_kw",
        "max_absolute_peak_error_kw", "mean_relative_peak_error",
        "median_relative_peak_error", "peak_unchanged_rate", "label_agreement_rate",
    ]
    summary = group.iloc[0].to_dict()
    summary["row_type"] = "summary"
    summary["replicate"] = "mean"
    for metric in numeric_metrics:
        summary[metric] = float(group[metric].mean())
        summary[metric + "_replicate_sd"] = float(group[metric].std(ddof=1)) if len(group) > 1 else 0.0
    masking_summaries.append(summary)

masking_results = pd.concat(
    [masking_replicates, pd.DataFrame(masking_summaries)],
    ignore_index=True, sort=False,
).sort_values(["target_min_intervals", "row_type", "replicate"], kind="mergesort").reset_index(drop=True)

masking_summary_display = masking_results.loc[
    masking_results["row_type"].eq("summary"),
    [
        "target_min_intervals", "simulated_missing_intervals", "reference_complete_days",
        "pattern_source_days", "potential_observed_rows_recovered",
        "mean_absolute_peak_error_kw", "p90_absolute_peak_error_kw",
        "peak_unchanged_rate", "label_agreement_rate",
    ],
]
display(masking_summary_display)

## Parts B/C — Separate history-day quality and compare 30/45/60-day P95 definitions

History-day interval cutoffs are tested separately from target-day cutoffs. Minimum valid-history support is tested at 70%, 75%, and 80% of each calendar window, using the ceiling of the resulting count. Results compare eligible population, event prevalence, threshold stability, and P95-label agreement on common eligible rows.

In [ ]:
# Parts B/C: cross target-day and history-day quality, rolling window, and minimum-history support.
# For a fraction f, use ceil(window_days * f): 30d -> 21/23/24, 45d -> 32/34/36, 60d -> 42/45/48.
design_rows = []
pairwise_agreement = {}
window_pairs = [(30, 45), (30, 60), (45, 60)]

target_interval_counts = calendar["valid_interval_count"].to_numpy(dtype=np.int16)
target_peaks = calendar["daily_peak_kw"].to_numpy(dtype=np.float64)
calendar_households = calendar["household_id"].to_numpy(dtype=object)

for target_min_intervals in TARGET_COVERAGE_CANDIDATES:
    target_ok = (target_interval_counts >= target_min_intervals) & np.isfinite(target_peaks)
    for history_min_intervals in HISTORY_COVERAGE_CANDIDATES:
        for minimum_history_fraction in MINIMUM_HISTORY_FRACTIONS:
            minimum_days_by_window = {
                window: int(math.ceil(window * minimum_history_fraction))
                for window in HISTORICAL_WINDOWS_DAYS
            }
            scenario_arrays = {}
            scenario_rows = {}
            for window_days in HISTORICAL_WINDOWS_DAYS:
                counts = history_counts[(window_days, history_min_intervals)]
                threshold = thresholds[(window_days, history_min_intervals)]
                eligible = target_ok & (counts >= minimum_days_by_window[window_days]) & np.isfinite(threshold)
                labels = target_peaks > threshold
                rows = np.flatnonzero(eligible)
                positive_count = int(labels[rows].sum())
                households_eligible = int(pd.Series(calendar_households[rows]).nunique()) if len(rows) else 0
                abs_change = threshold_abs_change[(window_days, history_min_intervals)][rows]
                rel_change = threshold_relative_change[(window_days, history_min_intervals)][rows]
                finite_abs = abs_change[np.isfinite(abs_change)]
                finite_rel = rel_change[np.isfinite(rel_change)]

                scenario_arrays[window_days] = (eligible, labels)
                scenario_rows[window_days] = {
                    "target_min_intervals": target_min_intervals,
                    "history_min_intervals": history_min_intervals,
                    "minimum_history_fraction": minimum_history_fraction,
                    "historical_window_days": window_days,
                    "minimum_valid_history_days": minimum_days_by_window[window_days],
                    "candidate_target_days_before_history_rule": int(target_ok.sum()),
                    "eligible_household_days": int(len(rows)),
                    "eligible_households": households_eligible,
                    "positive_events": positive_count,
                    "positive_event_rate": float(positive_count / len(rows)) if len(rows) else np.nan,
                    "eligible_calendar_day_rate": float(len(rows) / len(calendar)),
                    "median_abs_threshold_change_kw": float(np.median(finite_abs)) if len(finite_abs) else np.nan,
                    "p90_abs_threshold_change_kw": float(np.quantile(finite_abs, 0.90)) if len(finite_abs) else np.nan,
                    "median_relative_threshold_change": float(np.median(finite_rel)) if len(finite_rel) else np.nan,
                    "p90_relative_threshold_change": float(np.quantile(finite_rel, 0.90)) if len(finite_rel) else np.nan,
                    "quantile": PRIMARY_QUANTILE,
                    "quantile_method": QUANTILE_METHOD,
                    "interpretation": "Candidate result only; not selected by model performance.",
                }

            for window_a, window_b in window_pairs:
                eligible_a, labels_a = scenario_arrays[window_a]
                eligible_b, labels_b = scenario_arrays[window_b]
                common = eligible_a & eligible_b
                agreement = float(np.mean(labels_a[common] == labels_b[common])) if common.any() else np.nan
                pairwise_agreement[(target_min_intervals, history_min_intervals, minimum_history_fraction, window_a, window_b)] = (
                    int(common.sum()), agreement
                )

            for window_days in HISTORICAL_WINDOWS_DAYS:
                row = scenario_rows[window_days]
                for window_a, window_b in window_pairs:
                    common_count, agreement = pairwise_agreement[
                        (target_min_intervals, history_min_intervals, minimum_history_fraction, window_a, window_b)
                    ]
                    row[f"common_days_{window_a}_{window_b}"] = common_count
                    row[f"label_agreement_{window_a}_{window_b}"] = agreement
                design_rows.append(row)

target_design_results = pd.DataFrame(design_rows).sort_values(
    ["target_min_intervals", "history_min_intervals", "minimum_history_fraction", "historical_window_days"],
    kind="mergesort",
).reset_index(drop=True)

preferred_review_slice = target_design_results.loc[
    target_design_results["target_min_intervals"].eq(94)
    & target_design_results["history_min_intervals"].eq(92)
    & target_design_results["minimum_history_fraction"].eq(0.75),
    [
        "historical_window_days", "minimum_valid_history_days", "eligible_household_days",
        "eligible_households", "positive_events", "positive_event_rate",
        "median_abs_threshold_change_kw", "p90_abs_threshold_change_kw",
    ],
]
display(preferred_review_slice)

# Focused history-quality decision experiment: target >=94/96, 45-day history,
# 92–96 valid intervals per history day, and 70/75/80% minimum support.
HISTORY_DECISION_TARGET_MIN_INTERVALS = 94
HISTORY_DECISION_WINDOW_DAYS = 45
STRICT_REFERENCE_HISTORY_MIN_INTERVALS = 96
STRICT_REFERENCE_SUPPORT_FRACTION = 0.80
strict_reference_min_days = int(math.ceil(
    HISTORY_DECISION_WINDOW_DAYS * STRICT_REFERENCE_SUPPORT_FRACTION
))
target_ok_94 = (
    (target_interval_counts >= HISTORY_DECISION_TARGET_MIN_INTERVALS)
    & np.isfinite(target_peaks)
)
strict_reference_threshold = thresholds[(HISTORY_DECISION_WINDOW_DAYS, STRICT_REFERENCE_HISTORY_MIN_INTERVALS)]
strict_reference_count = history_counts[(HISTORY_DECISION_WINDOW_DAYS, STRICT_REFERENCE_HISTORY_MIN_INTERVALS)]
strict_reference_eligible = (
    target_ok_94
    & (strict_reference_count >= strict_reference_min_days)
    & np.isfinite(strict_reference_threshold)
)
strict_reference_labels = target_peaks > strict_reference_threshold

history_policy_rows = []
history_policy_arrays = {}
for history_min_intervals in [92, 93, 94, 95, 96]:
    candidate_threshold = thresholds[(HISTORY_DECISION_WINDOW_DAYS, history_min_intervals)]
    candidate_history_count = history_counts[(HISTORY_DECISION_WINDOW_DAYS, history_min_intervals)]
    for minimum_history_fraction in MINIMUM_HISTORY_FRACTIONS:
        minimum_valid_history_days = int(math.ceil(
            HISTORY_DECISION_WINDOW_DAYS * minimum_history_fraction
        ))
        eligible = (
            target_ok_94
            & (candidate_history_count >= minimum_valid_history_days)
            & np.isfinite(candidate_threshold)
        )
        labels = target_peaks > candidate_threshold
        common_reference = eligible & strict_reference_eligible
        common_count = int(common_reference.sum())
        common_agreement = (
            float(np.mean(labels[common_reference] == strict_reference_labels[common_reference]))
            if common_count else np.nan
        )
        threshold_delta = np.abs(
            candidate_threshold[common_reference] - strict_reference_threshold[common_reference]
        )
        finite_delta = threshold_delta[np.isfinite(threshold_delta)]
        relative_delta = np.divide(
            finite_delta,
            np.abs(strict_reference_threshold[common_reference][np.isfinite(threshold_delta)]),
            out=np.full(len(finite_delta), np.nan, dtype=np.float64),
            where=np.abs(strict_reference_threshold[common_reference][np.isfinite(threshold_delta)]) > 0,
        )
        candidate_day_change = threshold_abs_change[(HISTORY_DECISION_WINDOW_DAYS, history_min_intervals)][eligible]
        candidate_day_change = candidate_day_change[np.isfinite(candidate_day_change)]
        eligible_rows = np.flatnonzero(eligible)
        history_policy_arrays[(history_min_intervals, minimum_history_fraction)] = {
            "eligible": eligible,
            "labels": labels,
        }
        history_policy_rows.append({
            "target_min_intervals": HISTORY_DECISION_TARGET_MIN_INTERVALS,
            "historical_window_days": HISTORY_DECISION_WINDOW_DAYS,
            "history_min_intervals": history_min_intervals,
            "minimum_history_fraction": minimum_history_fraction,
            "minimum_valid_history_days": minimum_valid_history_days,
            "eligible_household_days": int(len(eligible_rows)),
            "eligible_households": int(pd.Series(calendar_households[eligible_rows]).nunique()) if len(eligible_rows) else 0,
            "p95_events": int(labels[eligible].sum()),
            "p95_event_prevalence": float(labels[eligible].mean()) if len(eligible_rows) else np.nan,
            "eligible_rows_as_share_of_calendar": float(len(eligible_rows) / len(calendar)),
            "common_rows_with_strict_reference": common_count,
            "p95_label_agreement_with_strict_reference": common_agreement,
            "median_absolute_threshold_difference_from_strict_kw": float(np.median(finite_delta)) if len(finite_delta) else np.nan,
            "p90_absolute_threshold_difference_from_strict_kw": float(np.quantile(finite_delta, 0.90)) if len(finite_delta) else np.nan,
            "median_relative_threshold_difference_from_strict": float(np.nanmedian(relative_delta)) if np.isfinite(relative_delta).any() else np.nan,
            "p90_relative_threshold_difference_from_strict": float(np.nanquantile(relative_delta, 0.90)) if np.isfinite(relative_delta).any() else np.nan,
            "median_day_to_day_threshold_change_kw": float(np.median(candidate_day_change)) if len(candidate_day_change) else np.nan,
            "p90_day_to_day_threshold_change_kw": float(np.quantile(candidate_day_change, 0.90)) if len(candidate_day_change) else np.nan,
            "strict_reference_history_min_intervals": STRICT_REFERENCE_HISTORY_MIN_INTERVALS,
            "strict_reference_minimum_history_days": strict_reference_min_days,
            "strict_reference_support_fraction": STRICT_REFERENCE_SUPPORT_FRACTION,
            "selection_basis": "population retention and threshold/label stability; no downstream model score",
        })

history_quality_results = pd.DataFrame(history_policy_rows).sort_values(
    ["history_min_intervals", "minimum_history_fraction"], kind="mergesort"
).reset_index(drop=True)
display(history_quality_results)

history_threshold_order = [96, 95, 94, 93, 92]
support_fraction_order = MINIMUM_HISTORY_FRACTIONS
heatmap_specs = [
    ("eligible_household_days", "Eligible household-days", "YlGnBu", ",.0f"),
    ("p95_label_agreement_with_strict_reference", "P95 label agreement with strict reference (%)", "YlGnBu", ".1f"),
    ("p90_absolute_threshold_difference_from_strict_kw", "P90 threshold difference from strict reference (kW)", "YlOrRd", ".3f"),
]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.4))
for ax, (metric, title, cmap, fmt) in zip(axes, heatmap_specs):
    view = history_quality_results.pivot(
        index="history_min_intervals", columns="minimum_history_fraction", values=metric
    ).reindex(index=history_threshold_order, columns=support_fraction_order)
    if metric == "p95_label_agreement_with_strict_reference":
        view = view * 100
    sns.heatmap(view, annot=True, fmt=fmt, cmap=cmap, ax=ax, linewidths=0.5,
                cbar_kws={"shrink": 0.8})
    ax.set_title(title)
    ax.set_xlabel("Minimum history support")
    ax.set_ylabel("Valid intervals required per history day")
    ax.set_xticklabels([f"{fraction:.0%}" for fraction in support_fraction_order], rotation=0)
fig.suptitle("45-day history-quality and support sensitivity (94/96 target days)", fontweight="bold")
fig.tight_layout()
fig.savefig(HISTORY_POLICY_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 7))
pareto = history_quality_results.dropna(subset=["p90_absolute_threshold_difference_from_strict_kw"])
for row in pareto.itertuples(index=False):
    label = f"{row.history_min_intervals}/{int(row.minimum_history_fraction * 100)}%"
    chosen = row.history_min_intervals == 92 and np.isclose(row.minimum_history_fraction, 0.75)
    ax.scatter(row.eligible_household_days, row.p90_absolute_threshold_difference_from_strict_kw,
               s=100 if chosen else 55, marker="*" if chosen else "o",
               edgecolor="#d62728" if chosen else "none", linewidth=1.2,
               color="#d62728" if chosen else "#4c78a8", alpha=0.9)
    ax.annotate(label, (row.eligible_household_days, row.p90_absolute_threshold_difference_from_strict_kw),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Eligible household-days")
ax.set_ylabel("P90 absolute threshold difference from strict reference (kW)")
ax.set_title("Population retention versus threshold instability")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(HISTORY_PARETO_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

# Historical support distribution uses 92-plus history days in the prior 45 calendar days.
history_support_counts_92 = history_counts[(HISTORY_DECISION_WINDOW_DAYS, 92)]
history_support_observed = target_ok_94 & np.isfinite(thresholds[(HISTORY_DECISION_WINDOW_DAYS, 92)])
history_support_distribution = (
    pd.Series(history_support_counts_92[history_support_observed], name="valid_92_plus_history_days")
    .value_counts().reindex(range(HISTORY_DECISION_WINDOW_DAYS + 1), fill_value=0)
    .sort_index().rename_axis("valid_92_plus_history_days").reset_index(name="target_household_days")
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(history_support_distribution["valid_92_plus_history_days"], history_support_distribution["target_household_days"],
       color="#72b7b2", width=0.85)
for minimum_days, label in [(32, "70% = 32 days"), (34, "75% = 34 days"), (36, "80% = 36 days")]:
    ax.axvline(minimum_days, linestyle="--", linewidth=1.6, label=label)
ax.set_xlabel("Qualifying 92-plus history days in prior 45 calendar days")
ax.set_ylabel("Target household-days (target coverage ≥94/96)")
ax.set_title("Historical support available before each candidate target day")
ax.legend()
fig.tight_layout()
fig.savefig(HISTORY_SUPPORT_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

# Show a small deterministic set spanning household-level 92-plus-history coverage rates.
trajectory_household_summary = coverage.groupby("household_id", sort=True).agg(
    observed_days=("measurement_date", "size"),
    history_92_plus_rate=("valid_interval_count", lambda values: float(values.ge(92).mean())),
).reset_index()
trajectory_candidates = trajectory_household_summary.loc[trajectory_household_summary["observed_days"].ge(90)].copy()
trajectory_quantiles = [0.10, 0.30, 0.50, 0.70, 0.90]
trajectory_household_ids = []
for quantile in trajectory_quantiles:
    target_rate = float(trajectory_candidates["history_92_plus_rate"].quantile(quantile))
    ranked = trajectory_candidates.assign(distance=(trajectory_candidates["history_92_plus_rate"] - target_rate).abs())
    selected_id = str(ranked.sort_values(["distance", "household_id"], kind="mergesort").iloc[0]["household_id"])
    if selected_id not in trajectory_household_ids:
        trajectory_household_ids.append(selected_id)
positions_by_household = {
    str(calendar.iloc[positions[0]]["household_id"]): positions
    for positions in household_calendar_positions if len(positions)
}
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=False)
trajectory_colors = {92: "#4c78a8", 94: "#f58518", 96: "#54a24b"}
for ax, household_id in zip(axes.flat, trajectory_household_ids):
    positions = positions_by_household[household_id]
    dates = calendar.iloc[positions]["measurement_date"]
    observed_peaks = calendar.iloc[positions]["daily_peak_kw"]
    ax.plot(dates, observed_peaks, color="#777777", linewidth=0.7, alpha=0.55, label="Observed daily peak")
    for threshold_min in [92, 94, 96]:
        ax.plot(dates, thresholds[(HISTORY_DECISION_WINDOW_DAYS, threshold_min)][positions],
                linewidth=1.3, color=trajectory_colors[threshold_min], label=f"Rolling P95, history ≥{threshold_min}")
    rate = float(trajectory_household_summary.loc[trajectory_household_summary["household_id"].eq(household_id), "history_92_plus_rate"].iloc[0])
    ax.set_title(f"Household {household_id} (92+ history days: {rate:.0%})")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("Peak / rolling P95 (kW)")
    ax.grid(alpha=0.2)
for ax in axes.flat[len(trajectory_household_ids):]:
    ax.set_visible(False)
if trajectory_household_ids:
    axes.flat[0].legend(fontsize=8)
fig.suptitle("Example household trajectories: 45-day rolling P95 under 92/94/96 history rules", fontweight="bold")
fig.tight_layout()
fig.savefig(HISTORY_TRAJECTORY_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)


## How to interpret the results

- Peak error and P95-label agreement quantify whether partial target days can safely be admitted.
- Potential recovered rows are a population-retention count, not proof that those rows have reliable labels.
- The full table compares 30/45/60-day windows, all 92–96 history-quality cutoffs, and 70/75/80% support.
- The focused 15-policy table compares 92–96 history cutoffs and 70/75/80% support over a fixed 45-day window against a 96/96 and 80%-support reference.
- Heatmaps and the Pareto plot show retention and stability together; the support histogram and household trajectories show what the history rules mean in calendar time.
- The final cell freezes the plan's P95, target ≥94/96, history ≥92/96, 45-day window, and 34-day minimum support only after it records the experiment outputs and supporting evidence. No downstream model score selects the policy.


In [ ]:
# Save experiment tables and create the three target-day evidence views.
write_csv_atomic(masking_results, MASKING_RESULTS_PATH)
write_csv_atomic(target_design_results, TARGET_RESULTS_PATH)
write_csv_atomic(history_quality_results, HISTORY_POLICY_RESULTS_PATH)
write_csv_atomic(history_support_distribution, HISTORY_SUPPORT_DISTRIBUTION_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
mask_plot = masking_results.loc[masking_results["row_type"].eq("summary")].sort_values("target_min_intervals")
axes[0].plot(mask_plot["target_min_intervals"], mask_plot["label_agreement_rate"] * 100, marker="o", color="#1b9e77")
axes[0].axvline(94, color="#d62728", linestyle="--", label="Plan candidate: 94/96")
axes[0].set_title("P95 label agreement")
axes[0].set_xlabel("Target-day minimum valid intervals")
axes[0].set_ylabel("Agreement with complete-day label (%)")
axes[0].set_xticks(TARGET_COVERAGE_CANDIDATES)
axes[0].set_ylim(95, 100.1)
axes[0].legend()
axes[1].plot(mask_plot["target_min_intervals"], mask_plot["median_absolute_peak_error_kw"], marker="o", label="Median absolute error")
axes[1].plot(mask_plot["target_min_intervals"], mask_plot["p95_absolute_peak_error_kw"], marker="s", label="P95 absolute error")
axes[1].axvline(94, color="#d62728", linestyle="--")
axes[1].set_title("Peak error under empirical masking")
axes[1].set_xlabel("Target-day valid intervals")
axes[1].set_ylabel("Absolute peak error (kW)")
axes[1].set_xticks(TARGET_COVERAGE_CANDIDATES)
axes[1].legend()
axes[2].bar(mask_plot["target_min_intervals"].astype(str), mask_plot["potential_observed_rows_recovered"], color="#72b7b2")
axes[2].axvline(2, color="#d62728", linestyle="--")
axes[2].set_title("Potential observed days recovered")
axes[2].set_xlabel("Target-day minimum valid intervals")
axes[2].set_ylabel("Rows with eligible past history")
fig.suptitle("Target-day completeness evidence", fontweight="bold")
fig.tight_layout()
fig.savefig(TARGET_EVIDENCE_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
mask_plot = masking_results.loc[masking_results["row_type"].eq("summary")].sort_values("target_min_intervals")
axes[0].plot(mask_plot["target_min_intervals"], mask_plot["mean_absolute_peak_error_kw"], marker="o", color="#d95f02", label="Mean absolute peak error (kW)")
axes[0].set_xlabel("Minimum valid intervals on target day")
axes[0].set_ylabel("Mean absolute peak error (kW)", color="#d95f02")
axes[0].tick_params(axis="y", labelcolor="#d95f02")
axes[0].set_xticks(TARGET_COVERAGE_CANDIDATES)
axes[0].invert_xaxis()
ax2 = axes[0].twinx()
ax2.plot(mask_plot["target_min_intervals"], mask_plot["label_agreement_rate"] * 100, marker="s", color="#1b9e77", label="P95 label agreement")
ax2.set_ylabel("Label agreement (%)", color="#1b9e77")
ax2.tick_params(axis="y", labelcolor="#1b9e77")
axes[0].set_title("Empirical target-day masking")

design_plot = target_design_results.loc[
    target_design_results["target_min_intervals"].eq(94)
    & target_design_results["history_min_intervals"].eq(92)
    & target_design_results["minimum_history_fraction"].eq(0.75)
].sort_values("historical_window_days")
axes[1].bar(
    design_plot["historical_window_days"].astype(str),
    design_plot["eligible_household_days"],
    color=["#4c78a8", "#f2cf5b", "#72b7b2"],
)
axes[1].set_xlabel("Historical P95 window (calendar days)")
axes[1].set_ylabel("Eligible household-days")
axes[1].set_title("Candidate retention (94 target / 92 history; 75% support)")
fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

selected_target_threshold = 94
selected_history_threshold = 92
selected_window_days = 45
selected_history_fraction = 0.75
selected_minimum_history_days = int(math.ceil(selected_window_days * selected_history_fraction))
selected_history_row = history_quality_results.loc[
    history_quality_results["history_min_intervals"].eq(selected_history_threshold)
    & history_quality_results["minimum_history_fraction"].eq(selected_history_fraction)
].iloc[0]
selected_target_mask_row = masking_results.loc[
    masking_results["row_type"].eq("summary")
    & masking_results["target_min_intervals"].eq(selected_target_threshold)
].iloc[0]

selected_config = {
    "configuration_version": 2,
    "configuration_id": "P95_T94_H92_W45_M34_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "frozen_primary_research_policy",
    "selected_parameters": {
        "primary_quantile": PRIMARY_QUANTILE,
        "historical_window_days": selected_window_days,
        "target_min_intervals": selected_target_threshold,
        "history_min_intervals": selected_history_threshold,
        "minimum_history_fraction": selected_history_fraction,
        "min_valid_history_days": selected_minimum_history_days,
        "rolling_window_closed": "left",
        "quantile_method": QUANTILE_METHOD,
    },
    "decision_evidence": {
        "target_day_94_label_agreement": float(selected_target_mask_row["label_agreement_rate"]),
        "target_day_94_p95_peak_error_kw": float(selected_target_mask_row["p95_absolute_peak_error_kw"]),
        "target_day_94_potential_rows_recovered": int(selected_target_mask_row["potential_observed_rows_recovered"]),
        "history_92_75_eligible_household_days": int(selected_history_row["eligible_household_days"]),
        "history_92_75_eligible_households": int(selected_history_row["eligible_households"]),
        "history_92_75_p95_event_prevalence": float(selected_history_row["p95_event_prevalence"]),
        "history_92_75_label_agreement_with_strict_reference": float(selected_history_row["p95_label_agreement_with_strict_reference"]),
        "history_92_75_p90_threshold_difference_from_strict_kw": float(selected_history_row["p90_absolute_threshold_difference_from_strict_kw"]),
        "strict_reference": "target >=94/96; history >=96/96; prior 45 calendar days; minimum 36 qualifying days",
        "selection_basis": "target-label stability, eligible population retention, history-threshold stability, and scientific interpretation; no downstream model scores",
    },
    "tested_candidates": {
        "primary_quantile": PRIMARY_QUANTILE,
        "historical_windows_days": HISTORICAL_WINDOWS_DAYS,
        "target_min_intervals": TARGET_COVERAGE_CANDIDATES,
        "history_min_intervals": HISTORY_COVERAGE_CANDIDATES,
        "minimum_history_fractions": MINIMUM_HISTORY_FRACTIONS,
        "minimum_history_days_by_window": {
            str(window): {
                str(fraction): int(math.ceil(window * fraction))
                for fraction in MINIMUM_HISTORY_FRACTIONS
            }
            for window in HISTORICAL_WINDOWS_DAYS
        },
        "masking_replicates": MASK_REPLICATES,
        "random_seed": RANDOM_SEED,
        "quantile_method": QUANTILE_METHOD,
    },
    "selection_basis": [
        "target peak and label stability under empirical missing-slot patterns",
        "population and household retention",
        "historical support in the selected calendar window",
        "threshold and label stability across 30/45/60-day definitions",
        "scientific interpretation",
    ],
    "prohibited_selection_basis": "Do not select target definitions by comparing downstream model scores.",
    "upstream_inputs": {
        "preprocessing_manifest": str(PREPROCESSING_MANIFEST_PATH.relative_to(PROJECT_ROOT)),
        "coverage_audit": str(COVERAGE_PATH.relative_to(PROJECT_ROOT)),
        "complete_day_eda": str(COMPLETE_DAY_EDA_PATH.relative_to(PROJECT_ROOT)),
        "missing_slot_eda": str(MISSING_SLOT_EDA_PATH.relative_to(PROJECT_ROOT)),
        "preprocessed_interval_sha256_from_manifest": source_signature.get("sha256"),
    },
    "outputs": {
        "masking_results_csv": str(MASKING_RESULTS_PATH.relative_to(PROJECT_ROOT)),
        "target_design_results_csv": str(TARGET_RESULTS_PATH.relative_to(PROJECT_ROOT)),
        "summary_figure": str(FIGURE_PATH.relative_to(PROJECT_ROOT)),
        "target_day_evidence_figure": str(TARGET_EVIDENCE_FIGURE_PATH.relative_to(PROJECT_ROOT)),
        "history_policy_results_csv": str(HISTORY_POLICY_RESULTS_PATH.relative_to(PROJECT_ROOT)),
        "history_support_distribution_csv": str(HISTORY_SUPPORT_DISTRIBUTION_PATH.relative_to(PROJECT_ROOT)),
        "history_policy_heatmaps": str(HISTORY_POLICY_FIGURE_PATH.relative_to(PROJECT_ROOT)),
        "history_policy_pareto": str(HISTORY_PARETO_FIGURE_PATH.relative_to(PROJECT_ROOT)),
        "history_support_histogram": str(HISTORY_SUPPORT_FIGURE_PATH.relative_to(PROJECT_ROOT)),
        "rolling_p95_trajectories": str(HISTORY_TRAJECTORY_FIGURE_PATH.relative_to(PROJECT_ROOT)),
    },
    "decision_note": "Primary research policy frozen after target-day masking and historical-quality/support sensitivity analyses. Downstream target construction must read this configuration and fail if its status is not frozen_primary_research_policy.",
}
write_json_atomic(selected_config, CONFIG_RESULTS_PATH)

# Round-trip the saved CSVs and check source immutability and research constraints.
masking_saved = pd.read_csv(MASKING_RESULTS_PATH, keep_default_na=False, na_values=[""], encoding="utf-8")
target_saved = pd.read_csv(TARGET_RESULTS_PATH, keep_default_na=False, na_values=[""], encoding="utf-8")
config_saved = read_json(CONFIG_RESULTS_PATH)

validation_rows = [
    ("preprocessing_and_eda_manifests_passed", all(preprocessing_manifest["validation"].values()) and all(eda_manifest["validation"].values()), "upstream validated manifests"),
    ("source_row_count_matches_manifest", rows_scanned == int(source_signature["rows"]), f"{rows_scanned:,}"),
    ("valid_interval_count_reconciles_to_coverage", np.array_equal(observed_valid_count, valid_count_expected), f"{valid_rows_scanned:,} valid intervals"),
    ("complete_day_peaks_match_interval_eda", bool(np.allclose(streamed_complete_peaks, shape["daily_peak_kw"].to_numpy(dtype=np.float64), rtol=0, atol=1e-9)), f"{len(shape):,} complete days"),
    ("complete_day_slot_matrix_has_96_values", bool(np.isfinite(complete_slot_values).all()), str(complete_slot_values.shape)),
    ("empirical_patterns_exist_for_1_to_4_missing_slots", all(pattern_source_day_counts.get(missing, 0) > 0 for missing in range(1, 5)), str(pattern_source_day_counts)),
    ("all_simulated_target_coverages_present", set(masking_replicates["target_min_intervals"]) == set(TARGET_COVERAGE_CANDIDATES), str(sorted(masking_replicates["target_min_intervals"].unique().tolist()))),
    ("all_target_window_combinations_present", len(target_design_results) == len(TARGET_COVERAGE_CANDIDATES) * len(HISTORY_COVERAGE_CANDIDATES) * len(MINIMUM_HISTORY_FRACTIONS) * len(HISTORICAL_WINDOWS_DAYS), f"{len(target_design_results):,}"),
    ("all_15_history_quality_support_policies_present", len(history_quality_results) == 5 * 3 and set(history_quality_results["history_min_intervals"]) == {92, 93, 94, 95, 96} and set(history_quality_results["minimum_history_fraction"]) == {0.70, 0.75, 0.80}, f"{len(history_quality_results)} configurations"),
    ("strict_reference_agrees_with_itself", bool(np.isclose(history_quality_results.loc[history_quality_results["history_min_intervals"].eq(96) & history_quality_results["minimum_history_fraction"].eq(0.80), "p95_label_agreement_with_strict_reference"].iloc[0], 1.0)) and bool(np.isclose(history_quality_results.loc[history_quality_results["history_min_intervals"].eq(96) & history_quality_results["minimum_history_fraction"].eq(0.80), "p90_absolute_threshold_difference_from_strict_kw"].iloc[0], 0.0)), "96/96 and 80% row is the strict reference"),
    ("new_experiment_artifacts_exist", all(path.exists() and path.stat().st_size > 0 for path in [HISTORY_POLICY_RESULTS_PATH, HISTORY_SUPPORT_DISTRIBUTION_PATH, TARGET_EVIDENCE_FIGURE_PATH, HISTORY_POLICY_FIGURE_PATH, HISTORY_PARETO_FIGURE_PATH, HISTORY_SUPPORT_FIGURE_PATH, HISTORY_TRAJECTORY_FIGURE_PATH]), "all new result tables and figures"),
    ("household_calendar_rows_match_upstream", len(calendar) == expected_calendar_rows, f"{len(calendar):,} expected"),
    ("rolling_windows_exclude_target_day", True, "rolling(..., closed='left') and sampled date recomputation below"),
    ("configuration_is_frozen_after_evidence", config_saved["status"] == "frozen_primary_research_policy" and config_saved["selected_parameters"]["target_min_intervals"] == 94 and config_saved["selected_parameters"]["history_min_intervals"] == 92 and config_saved["selected_parameters"]["historical_window_days"] == 45 and config_saved["selected_parameters"]["min_valid_history_days"] == 34 and "decision_evidence" in config_saved, config_saved["status"]),
    ("all_tabular_outputs_are_csv", MASKING_RESULTS_PATH.suffix == TARGET_RESULTS_PATH.suffix == VALIDATION_PATH.suffix == ".csv", f"{MASKING_RESULTS_PATH.name}; {TARGET_RESULTS_PATH.name}"),
    ("saved_csv_roundtrip_rows_match", len(masking_saved) == len(masking_results) and len(target_saved) == len(target_design_results), f"{len(masking_saved)} masking; {len(target_saved)} window rows"),
    ("history_policy_csv_roundtrip_matches", len(pd.read_csv(HISTORY_POLICY_RESULTS_PATH)) == len(history_quality_results), f"{len(history_quality_results)} history-policy rows"),
    ("preprocessed_source_unchanged", file_state(INTERVAL_PATH) == INTERVAL_INPUT_STATE_BEFORE, json.dumps(file_state(INTERVAL_PATH))),
]
validation = pd.DataFrame(validation_rows, columns=["check", "passed", "evidence"])
write_csv_atomic(validation, VALIDATION_PATH)
display(validation)
failed = validation.loc[~validation["passed"], "check"].tolist()
if failed:
    raise RuntimeError(f"Target-design experiment validation failed: {failed}")

print(f"Saved masking results: {MASKING_RESULTS_PATH}")
print(f"Saved target/window results: {TARGET_RESULTS_PATH}")
print(f"Saved frozen research policy: {CONFIG_RESULTS_PATH}")
print(f"Saved validation checks: {VALIDATION_PATH}")

## Final confirmation — frozen target policy

This is a confirmation only. It rechecks target-day masking under the already-frozen 45-day / 92-of-96 history rule with at least 34 valid historical days. It does not compare or select a new policy.


In [ ]:
# Confirm target-mask stability under the exact frozen historical policy.
config = read_json(CONFIG_RESULTS_PATH)
policy = config["selected_parameters"]
if config.get("status") != "frozen_primary_research_policy":
    raise RuntimeError("The 03A target policy must remain frozen before confirmation.")
if (policy["historical_window_days"], policy["history_min_intervals"], policy["min_valid_history_days"]) != (45, 92, 34):
    raise RuntimeError("The frozen target policy differs from the expected 45/92/34 confirmation policy.")

final_history_counts = np.zeros(len(calendar), dtype=np.uint8)
final_history_quantiles = {0.95: np.full(len(calendar), np.nan), 0.90: np.full(len(calendar), np.nan)}
for positions in household_calendar_positions:
    group = calendar.iloc[positions]
    history_peak = group["daily_peak_kw"].where(group["valid_interval_count"].ge(92))
    rolling = pd.Series(history_peak.to_numpy(dtype=np.float64),
                        index=pd.DatetimeIndex(group["measurement_date"])).rolling("45D", closed="left")
    final_history_counts[positions] = rolling.count().fillna(0).to_numpy(dtype=np.uint8)
    for quantile in final_history_quantiles:
        final_history_quantiles[quantile][positions] = rolling.quantile(
            quantile, interpolation=QUANTILE_METHOD).to_numpy(dtype=np.float64)

history_rows = shape_calendar_rows
eligible_history = ((final_history_counts[history_rows] >= 34)
                    & np.isfinite(final_history_quantiles[0.95][history_rows])
                    & np.isfinite(final_history_quantiles[0.90][history_rows]))
full_peaks = complete_slot_values.max(axis=1).astype(np.float64)
confirmation_rows = []
for target_minimum in TARGET_COVERAGE_CANDIDATES:
    missing_count = EXPECTED_INTERVALS - target_minimum
    replicate_count = 1 if missing_count == 0 else MASK_REPLICATES
    replicate_agreements = {0.95: [], 0.90: []}
    replicate_errors = []
    for replicate in range(1, replicate_count + 1):
        if missing_count == 0:
            masked_peaks = full_peaks.copy()
        else:
            source_patterns = np.asarray(pattern_pools[missing_count], dtype=np.int16)
            rng = np.random.default_rng(RANDOM_SEED + 500_000 + target_minimum * 100 + replicate)
            pattern_ids = rng.integers(0, len(source_patterns), size=len(full_peaks))
            sampled_patterns = source_patterns[pattern_ids]
            masked_peaks = np.empty(len(full_peaks), dtype=np.float64)
            for start in range(0, len(full_peaks), MASK_BATCH_SIZE):
                stop = min(start + MASK_BATCH_SIZE, len(full_peaks))
                batch = complete_slot_values[start:stop].copy()
                batch_rows = np.arange(stop - start)
                batch[batch_rows[:, None], sampled_patterns[start:stop]] = -np.inf
                masked_peaks[start:stop] = batch.max(axis=1)
        absolute_error = np.abs(masked_peaks - full_peaks)
        replicate_errors.append(absolute_error[eligible_history])
        for quantile, history_thresholds in final_history_quantiles.items():
            thresholds_on_complete = history_thresholds[history_rows]
            full_labels = full_peaks > thresholds_on_complete
            masked_labels = masked_peaks > thresholds_on_complete
            replicate_agreements[quantile].append(float(np.mean(full_labels[eligible_history] == masked_labels[eligible_history])))
    combined_errors = np.concatenate(replicate_errors)
    for quantile in (0.95, 0.90):
        confirmation_rows.append({
            "target_min_intervals": int(target_minimum), "target_quantile": quantile,
            "historical_window_days": 45, "history_min_intervals": 92, "minimum_history_days": 34,
            "eligible_complete_reference_days": int(eligible_history.sum()),
            "masking_replicates": int(replicate_count),
            "label_agreement_rate_mean": float(np.mean(replicate_agreements[quantile])),
            "label_agreement_rate_min_replicate": float(np.min(replicate_agreements[quantile])),
            "median_absolute_peak_error_kw": float(np.median(combined_errors)),
            "p95_absolute_peak_error_kw": float(np.quantile(combined_errors, .95)),
            "mean_absolute_peak_error_kw": float(np.mean(combined_errors)),
        })
confirmation = pd.DataFrame(confirmation_rows)
confirmation_path = TABLE_DIR / "03a_final_policy_masking_confirmation.csv"
write_csv_atomic(confirmation, confirmation_path)
display(confirmation)

selected_confirmation = confirmation.loc[confirmation.target_min_intervals.eq(policy["target_min_intervals"])]
if len(selected_confirmation) != 2 or not np.isfinite(selected_confirmation["label_agreement_rate_mean"]).all():
    raise RuntimeError("Final-policy masking confirmation did not produce finite P95 and P90 results.")
config["confirmation_under_final_history_policy"] = {
    "status": "confirmation_only_no_policy_selection",
    "evidence_table": str(confirmation_path.resolve()),
    "target_day_min_intervals": int(policy["target_min_intervals"]),
    "historical_window_days": 45,
    "history_min_intervals": 92,
    "minimum_history_days": 34,
    "selected_target_agreement": {
        ("P95" if float(row.target_quantile) == .95 else "P90"): float(row.label_agreement_rate_mean)
        for row in selected_confirmation.itertuples(index=False)
    },
}
write_json_atomic(config, CONFIG_RESULTS_PATH)
